## BachGD : 
* 1 update after lookup of whole data upto epochs time (1update per epoch -> slow) {small, conved data}
## StochasticGD : 
* n updates per epoch -> fast {Big Data, non-convex} -> random

## MiniBatchGD :
* batch -> group of rows e.g n = 1000, rows = 100 -> 10 batches -> 10 update per epoch

In [1]:
from sklearn.datasets import load_diabetes

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

In [3]:
X, y = load_diabetes(return_X_y=True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [5]:
reg = LinearRegression()
reg.fit(X_train, y_train)

LinearRegression()

In [6]:
print(reg.coef_)
print(reg.intercept_)

[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
151.88331005254167


In [7]:
y_pred = reg.predict(X_test)

In [8]:
print(r2_score(y_test, y_pred))

0.439933866156897


In [24]:
import random
class Mini_BatchGDR:
    def __init__(self, batch_size, lr = 0.01, epochs = 100):
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train, y_train):
        # init
        self.intercept_ = 0
        self.coef_ = np.ones(X_train.shape[1])

        for i in range(self.epochs):                                  # epochs times
            for j in range(int(X_train.shape[0]/self.batch_size)):    # batch times
                idx = random.sample(range(X_train.shape[0]), self.batch_size)

                y_hat = np.dot(X_train[idx], self.coef_) + self.intercept_

                intercept_der = -2*np.mean(y_train[idx] - y_hat)
                self.intercept_ = self.intercept_ - (self.lr * intercept_der)

                coef_der = -2*np.dot((y_train[idx] - y_hat), X_train[idx])
                self.coef_ = self.coef_ - (self.lr * coef_der)
                
        print(self.coef_, self.intercept_)

    def predict(self, X_test):
        return np.dot(X_test, self.coef_) + self.intercept_

In [29]:
mbgdr = Mini_BatchGDR(int(X_train.shape[0]/10), 0.01, 100)
mbgdr.fit(X_train, y_train)

[  23.85581125 -145.00584519  458.02775139  294.60314091  -22.81277566
  -93.95424848 -189.64389614  109.65544475  411.68374206  114.0505112 ] 150.9462441798255


In [30]:
y_pred = mbgdr.predict(X_test)
print(r2_score(y_test, y_pred))

0.4546639664137201


## using sklearn

In [31]:
from sklearn.linear_model import SGDRegressor

In [43]:
model = SGDRegressor(learning_rate='constant', eta0 = 0.1)

In [44]:
batch_size = 35
for i in range(100):    # explicit epoch = 100 because internally partial_fit() has max_iter = 1
    idx = random.sample(range(X_train.shape[0]), batch_size)
    model.partial_fit(X_train[idx], y_train[idx])

In [45]:
model.coef_

array([  60.61047003,  -53.15052574,  329.41910752,  244.9000688 ,
         13.71705626,  -32.00620274, -174.63487747,  127.12171876,
        315.09259196,  123.90953158])

In [46]:
model.intercept_

array([150.97494024])

In [47]:
y_pred = model.predict(X_test)

In [48]:
print(r2_score(y_test, y_pred))

0.4277889734231215
